In [15]:
import pandas as pd

labeled_orders = pd.read_parquet(
    "data/artifacts/02_labeled_orders.parquet"
)

labeled_orders["order_purchase_timestamp"] = pd.to_datetime(
    labeled_orders["order_purchase_timestamp"]
)

labeled_orders = labeled_orders.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

print("Shape:", labeled_orders.shape)

Shape: (96476, 42)


In [16]:
n = len(labeled_orders)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train = labeled_orders.iloc[:train_end].copy()
validation = labeled_orders.iloc[train_end:validation_end].copy()
test = labeled_orders.iloc[validation_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 42)
Validation: (14471, 42)
Test: (14472, 42)


In [17]:
for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n===== {name} =====")
    print("من:", df["order_purchase_timestamp"].min())
    print("إلى:", df["order_purchase_timestamp"].max())


===== Train =====
من: 2016-09-15 12:16:38
إلى: 2018-04-15 20:07:56

===== Validation =====
من: 2018-04-15 20:10:23
إلى: 2018-06-21 07:50:39

===== Test =====
من: 2018-06-21 08:29:29
إلى: 2018-08-29 15:00:37


In [18]:
for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n===== {name} =====")
    print("عدد الصفوف:", len(df))

    print("\nالتوزيع:")
    print(df["is_delayed"].value_counts())

    print("\nالنسب المئوية:")
    print(
        df["is_delayed"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


===== Train =====
عدد الصفوف: 67533

التوزيع:
is_delayed
0    61436
1     6097
Name: count, dtype: int64

النسب المئوية:
is_delayed
0    90.97
1     9.03
Name: proportion, dtype: float64

===== Validation =====
عدد الصفوف: 14471

التوزيع:
is_delayed
0    13698
1      773
Name: count, dtype: int64

النسب المئوية:
is_delayed
0    94.66
1     5.34
Name: proportion, dtype: float64

===== Test =====
عدد الصفوف: 14472

التوزيع:
is_delayed
0    13515
1      957
Name: count, dtype: int64

النسب المئوية:
is_delayed
0    93.39
1     6.61
Name: proportion, dtype: float64


In [19]:
train.to_parquet(
    "data/artifacts/03_train.parquet",
    index=False
)

validation.to_parquet(
    "data/artifacts/03_validation.parquet",
    index=False
)

test.to_parquet(
    "data/artifacts/03_test.parquet",
    index=False
)

print("تم حفظ Train / Validation / Test بنجاح.")

تم حفظ Train / Validation / Test بنجاح.


In [6]:
train = pd.read_parquet("data/artifacts/03_train.parquet")
validation = pd.read_parquet("data/artifacts/03_validation.parquet")
test = pd.read_parquet("data/artifacts/03_test.parquet")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 42)
Validation: (14471, 42)
Test: (14472, 42)


In [7]:
for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n===== {name} =====")
    print("من:", df["order_purchase_timestamp"].min())
    print("إلى:", df["order_purchase_timestamp"].max())


===== Train =====
من: 2016-09-15 12:16:38
إلى: 2018-04-15 20:07:56

===== Validation =====
من: 2018-04-15 20:10:23
إلى: 2018-06-21 07:50:39

===== Test =====
من: 2018-06-21 08:29:29
إلى: 2018-08-29 15:00:37


In [8]:
for name, df in {
    "Train": train,
    "Validation": validation,
    "Test": test
}.items():

    print(f"\n===== {name} =====")
    print(df["is_delayed"].value_counts())
    print(
        df["is_delayed"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


===== Train =====
is_delayed
0    61436
1     6097
Name: count, dtype: int64
is_delayed
0    90.97
1     9.03
Name: proportion, dtype: float64

===== Validation =====
is_delayed
0    13698
1      773
Name: count, dtype: int64
is_delayed
0    94.66
1     5.34
Name: proportion, dtype: float64

===== Test =====
is_delayed
0    13515
1      957
Name: count, dtype: int64
is_delayed
0    93.39
1     6.61
Name: proportion, dtype: float64


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


train = pd.read_parquet(
    "data/artifacts/03_train.parquet"
)

print("Shape:", train.shape)
display(train.head())

Shape: (67533, 42)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_lat,customer_lng,...,total_product_volume_cm3,avg_product_photos_qty,avg_product_name_length,avg_product_description_length,seller_customer_distance_km,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_delayed
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,-20.585396,-47.863156,...,12288.0,1.0,34.0,1036.0,566.040211,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,1
1,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,-23.581321,-46.635726,...,4096.0,3.0,63.0,1642.0,708.535291,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,0
2,be5bc2f0da14d8071e2d45451ad119d9,7ec40b22510fdbea1b08921dd39e63d8,delivered,2016-10-03 16:56:50,2016-10-06 16:03:44,2016-10-21 16:33:46,2016-10-27 18:19:38,2016-11-07,-28.291275,-53.501401,...,4096.0,1.0,39.0,518.0,915.734331,2f64e403852e6893ae37485d5fcacdaf,98280,panambi,RS,0
3,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25,-22.936855,-43.359961,...,3332.0,1.0,25.0,823.0,358.782919,b8b8726af116a5cfb35b0315ecef9172,22770,rio de janeiro,RJ,0
4,a41c8759fbe7aab36ea07e038b2d4465,6f989332712d3222b6571b1cf5b835ce,delivered,2016-10-03 21:13:36,2016-10-05 03:11:49,2016-10-25 11:57:59,2016-11-03 10:58:07,2016-11-29,-30.040958,-51.212970,...,4160.0,1.0,39.0,141.0,818.048765,61db744d2f835035a5625b59350c6b63,90040,porto alegre,RS,0
